# Adjoint capture analysis: a stream, a lake, and the sea

A well pumping near the coast competes for water with every other outlet the
aquifer has. Some of what it produces is released from aquifer storage, and the
rest is **capture** — water that would otherwise have discharged to a stream, a
lake, or the sea, or that those features now lose to the aquifer. This notebook
splits the capture of one well three ways, using **adjoint-state sensitivity
analysis** with [mf6adj](https://github.com/INTERA-Inc/mf6adj) on the coastal
synthetic valley.

By the end of this notebook you will be able to:

- represent a saltwater coast in a constant-density model with an **equivalent
  freshwater head**,
- write one mf6adj **performance measure** per competing outlet and solve them
  all from a single backward solve,
- read the capture fraction each outlet supplies, and map it for a well placed
  anywhere in the aquifer,
- find where in the valley the sea replaces the stream as the main source of a
  well's water,
- measure how much more seawater the coast delivers for 0.3 m (about a foot) of
  sea-level rise, and
- check every measure with a small-perturbation two-run difference and account
  for the whole of the pumping in the water budget.

## The three outlets and one backward solve

The direct way to find out how sensitive a model result is to a parameter is to
change the parameter and run the model again. That costs one run per parameter,
so a sensitivity map over a 5-layer, 40-by-25 grid would cost thousands of runs.

The adjoint approach turns the problem around. Name the single model output you
care about — the **performance measure** — and solve the flow equations backward
in time once. That one backward solve returns the sensitivity of that measure to
every parameter in every cell: hydraulic conductivity, storage, recharge,
boundary conductance, and well rates. The cost is one backward solve per
measure, rather than one forward run per parameter, so three outlets cost three
backward solves and one forward run, whatever the size of the grid.

The sensitivity of a boundary flow to a well rate is the **capture fraction**
directly: it is the change in that boundary's flow per unit of pumping, so a
value of 1 means every unit pumped is taken from that feature and 0 means none of
it is.

mf6adj drives MODFLOW 6 through its Application Programming Interface (**API**)
to collect the matrices it needs, so the model itself is unmodified. The method
and its verification are described in Hayek and others (2025), *MF6-ADJ: A
Non-Intrusive Adjoint Sensitivity Capability for MODFLOW 6*, Groundwater 63(6),
874–888. [mf6-adj-capture](mf6-adj-capture.ipynb) works through the same method
on the inland valley, with two outlets instead of three.

Import the packages this notebook uses. `mf6adj` provides the adjoint solver and
`mf6_adj_helpers` collects the workspace setup and the sensitivity readers shared
by the adjoint notebooks. `%matplotlib inline` has to come first, or only the
first figure drawn inside a `flopy.plot.styles` context is rendered.

In [ ]:
%matplotlib inline

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from mf6_notebook_helpers import (
    DENSITY_SLOPE,
    FRESHWATER_DENSITY,
    SEAWATER_CONCENTRATION,
    find_mf6_libraries,
    length_name,
    to_display,
    unit,
)

Locate the MODFLOW 6 shared library and executable in the active environment.
mf6adj drives the model through the shared library (`libmf6`), so both paths are
needed: the executable for the ordinary forward runs and the library for the
adjoint.

In [ ]:
lib_name, mf6_exe = find_mf6_libraries()
print(f"library:    {lib_name.name}")
print(f"executable: {mf6_exe.name}")

## Prepare the coastal model

Use the **coastal advanced** synthetic-valley model at an annual sampling
frequency. It represents the valley with the advanced hydrologic packages:
streamflow routing (**SFR**) for the river, a lake (**LAK**) with no outlet,
unsaturated-zone flow (**UZF**) for infiltration and evapotranspiration above
the water table, seepage drains (**DRN**) that remove groundwater where the water
table reaches the land surface, and the water mover (**MVR**), which carries UZF
runoff and drain water to the lake or the stream. A general-head boundary
(**GHB**) along the southern row is the sea, built in
[mf6-coastal-ghb](mf6-coastal-ghb.ipynb). The 21 stress periods start with a
long steady spin-up followed by 20 annual periods, and the model is in meters and
days.

The model needs no preparation for the adjoint. The production wells are
multi-aquifer wells (**MAW**), whose heads MODFLOW 6 solves along with the
aquifer heads, so each well is already one of the rows of the matrix mf6adj
differentiates. MODFLOW 6 solves SFR and LAK separately from the aquifer heads
and passes the two back and forth until they agree, so mf6adj borders the adjoint
system with their own equations. UZF and the mover are not differentiated at
all, which is where the warnings further down come from.

`prepare_model()` copies the shipped model into `models/`, sets the prediction
well to a rate of zero, and raises the coastal boundary heads to equivalent
freshwater heads.

In [ ]:
ws = adjh.prepare_model(
    "adj-coastal-capture",
    variant="coastal-advanced",
    prediction_rate=0.0,
    equivalent_freshwater=True,
)
adjh.run_model(ws, mf6_exe)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(ws), verbosity_level=0)
gwf = sim.get_model()
nper = sim.tdis.nper.data
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
print(f"grid:           {nlay} layers, {nrow} rows, {ncol} columns")
print(f"stress periods: {nper}")
print(f"packages:       {', '.join(sorted(gwf.package_names))}")

This first run is the **baseline**: the prediction well is present but pumps at a
rate of zero. The sensitivities describe how the model responds to a small change
away from this baseline, which makes the capture fraction a property of the
aquifer and the well's position rather than of one particular pumping rate.

### Why the boundary head is not zero

Seawater is denser than fresh groundwater, so a column of it presses down harder
than the same column of fresh water would. This model carries a single constant
fluid density, and the way to represent the weight of the sea in such a model is
to raise the boundary head until a column of fresh water would press down just as
hard. That raised head is the **equivalent freshwater head**, and it grows with
depth below sea level:

```
h_f = h_s + (rho_s / rho_f - 1) * (h_s - z)
```

where `h_s` is sea level, `z` is the elevation of the boundary, and the density
ratio `rho_s / rho_f` is 1,024.5 over 1,000 for seawater at 35 kilograms per
cubic meter. Print what that gives in each layer. Leaving the boundary at sea
level everywhere would understate the resistance the sea offers to outflow, and
the understatement grows with depth.

In [ ]:
ghb_spd = gwf.get_package("ghb-1").stress_period_data.get_data(0)
# (layer, row, column), boundary head, conductance, elevation, concentration
print(f"first boundary: {tuple(ghb_spd[0])}")
print()
lu, cu = unit(plain=True), unit("conductance", plain=True)
print(f"layer   cells   elevation, {lu}     boundary head, {lu}   conductance, {cu}")
for k in range(nlay):
    sel = ghb_spd[[cellid[0] == k for cellid in ghb_spd["cellid"]]]
    if len(sel) == 0:
        print(f"  {k + 1}        0")
        continue
    elevation, bhead = to_display(sel["elevation"]), to_display(sel["bhead"])
    print(
        f"  {k + 1}      {len(sel):3d}   "
        f"{elevation.min():8.1f} to {elevation.max():7.1f}"
        f"    {bhead.min():6.3f} to {bhead.max():6.3f}"
        f"     {to_display(sel['cond'].sum(), 'conductance'):12,.0f}"
    )
total_cond = to_display(ghb_spd["cond"].sum(), "conductance")
print(f"\ntotal coastal conductance: {total_cond:,.0f} {cu}")

**What to look for.** The boundary head rises from 0.019 m (0.061 ft) at the
layer-1 boundaries, 0.8 m (2.5 ft) below sea level, to between 0.94 and 1.76 m
(3.08 and 5.77 ft) in layer 5, where the boundaries sit 38.4 to 71.8 m (126 to
236 ft) below sea level. Layer 3 has no coastal cells: the clay does not reach the
southern row, and elsewhere layer 3 has no thickness. Every one of those heads is
above sea level, so the sea resists outflow more than a zero head would and can
drive water inland where the aquifer head is low enough. Layers 4 and 5 carry
207,296 of the 233,047 m²/d (2,231,305 of 2,508,497 ft²/d) of coastal
conductance, or 88.9 percent of it, because the two deep layers are both thick
and permeable.

### Look at the model

Map the features that compete for the well's water: the lake cells, the stream
reaches, the seepage drains, the coastal boundary along the southern row, the two
production wells, and the prediction well. The prediction well's cell is read
from the prediction well package rather than typed in. Cell IDs are
`(layer, row, column)` counting from zero, as everywhere in FloPy.

In [ ]:
pred_cell = adjh.package_cells(gwf, "prediction")[0]  # zero-based (layer, row, column)
prod_cells = adjh.package_cells(gwf, "pwell")
sfr_cells = adjh.package_cells(gwf, "sfr-1")
lak_cells = adjh.package_cells(gwf, "lak-1")
drn_cells = adjh.package_cells(gwf, "drn-1")
coast_cells = adjh.package_cells(gwf, "ghb-1")
print(f"prediction well cell: {pred_cell}")

xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
px, py = xc[pred_cell[1], pred_cell[2]], yc[pred_cell[1], pred_cell[2]]
print("distance from the prediction well to the nearest")
for label, cells in (
    ("coast", coast_cells),
    ("stream reach", sfr_cells),
    ("lake cell", lak_cells),
):
    d = min(np.hypot(xc[i, j] - px, yc[i, j] - py) for _, i, j in cells)
    print(f"  {label:13s} {to_display(d):7,.0f} {unit(plain=True)}")

# 1 seepage drain, 2 lake, 3 stream, 4 coast
features = np.zeros((nrow, ncol))
for cells, value in ((drn_cells, 1), (lak_cells, 2), (sfr_cells, 3), (coast_cells, 4)):
    for _, i, j in cells:
        features[i, j] = value
feature_colors = ["wheat", "tab:blue", "tab:cyan", "tab:green"]
feature_labels = ["seepage drains", "lake", "stream", "coast"]

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(4.5, 7), layout="constrained")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    mm.plot_array(
        np.ma.masked_equal(features, 0),
        cmap=mpl.colors.ListedColormap(feature_colors),
        vmin=0.5,
        vmax=4.5,
    )
    mm.plot_grid(lw=0.2, color="0.6")
    for _, i, j in prod_cells:
        ax.plot(xc[i, j], yc[i, j], "ko", ms=7)
    ax.plot(px, py, "r*", ms=16, label="prediction well")
    ax.plot([], [], "ko", ms=7, label="production wells")
    for color, label in zip(feature_colors, feature_labels):
        ax.plot([], [], "s", color=color, mec="0.4", label=label)
    ax.legend(loc="upper right", fontsize=8)
    # label the axes in display units; the model grid stays in meters
    ax.xaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
    ax.yaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
    ax.set_xlabel(f"x, in {length_name()}")
    ax.set_ylabel(f"y, in {length_name()}")
    ax.set_title("Coastal synthetic valley: the outlets and the wells")

**What to look for.** The lake fills the northwest of the valley up to its
northern edge, the stream runs south from below it, and the coast is the
southern row, broken only at the column the stream leaves through. Seepage drains
cover the rest of layer 1. The prediction well (red star) is in layer 5, 762 m
(2,500 ft) from the coast, 1,067 m (3,500 ft) from the nearest stream reach, and
3,002 m (9,849 ft) from the nearest lake cell. The sea is the closest outlet to
this well and the lake the farthest.

## Define one performance measure per outlet

An mf6adj performance measure lists the model outputs to combine. mf6adj reads
it from a plain-text block with one line per entry:

```
# kper kstp layer row column  package  form  weight  observed
  21    1     5    40    12   ghb-1   direct  1.0    -1.0e+30
```

The `package` field is either `head` — for the head in that cell — or the name of
a boundary package from the model name file, which selects the flow between that
package and the cell. `direct` means the measure is the value itself rather than a
residual against an observation, so the `observed` field is unused.

Give `mf6adj.write_performance_measures()` each measure as a dictionary of
columns: the cells in `cellid`, exactly as FloPy returns them, and the time in
`kper` and `kstp`. All of these are zero-based, and the writer converts them to
the one-based indices of the file. The file is written as ascii text unless you
pass `format="hdf5"`.

Build three measures at the last stress period, each summing the exchange
between one package and the aquifer over all of that package's cells. Together
they are the surface water and seawater that pumping can capture. The seepage
drains are an outlet too, but the water budget at the end of the notebook shows
they carry no water in the final period, so they need no measure.

In [ ]:
last = nper - 1  # zero-based
measures = {}
for name, package, cells in (
    ("swgw", "sfr-1", sfr_cells),
    ("lakegw", "lak-1", lak_cells),
    ("coastgw", "ghb-1", coast_cells),
):
    print(f"{name:8s} {package:6s} {len(cells):4d} cells")
    measures[name] = {
        "cellid": cells,
        "kper": last,
        "kstp": 0,  # every period is a single time step
        "pm_type": package,
    }
adj_file = mf6adj.write_performance_measures(ws / "capture.adj", measures)
print()
print(adj_file.read_text()[:200] + "...")

## Solve the forward and adjoint problems

Create the `mf6adj.Mf6Adj` object with the measure file and the shared library.
`solve_forward_model()` runs MODFLOW 6 through the API and saves the solution
matrix and boundary terms at every time step. `solve_adjoint()` then sweeps
backward through those saved time steps, once per measure, and all three measures
come out of that single forward run. Always call `finalize()` to release the
library. `logging_level="WARNING"` keeps the per-time-step progress messages out
of the notebook.

Keep the level at `WARNING` rather than raising it, because that is where mf6adj
reports that part of a package's derivative is not formed. This model draws
warnings about UZF, the mover, and the lake.

In [ ]:
adj = mf6adj.Mf6Adj(
    adj_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(ws),
)
adj.solve_forward_model()
sensitivities = adj.solve_adjoint()
adj.finalize()

print(f"measures solved: {', '.join(sensitivities)}")
print(f"parameters:      {', '.join(sensitivities['coastgw'].columns)}")

**What to look for.** The first warning names two packages mf6adj forms no terms
for at all, UZF and the water mover. Their exchange with the aquifer is carried
by the flow matrix, so the head sensitivities account for it, but neither package
can be measured and neither one's response to pumping enters a derivative. The
second warning, repeated once for each measure, names the inflow the mover routes
into the lake, which follows the state of the package supplying it. Neither the
stream nor the coast draws a warning of its own.

Each measure returns a table with one row per model cell and one column per
parameter — hydraulic conductivity (`k11`, `k33`), the two storage properties
(specific storage `ss` and specific yield `sy`), recharge, the elevation and
conductance of the seepage drains, the stage and conductance of the stream and
the lake, the boundary head and conductance of the coast (`ghb-1_bhead`,
`ghb-1_cond`), and the well rate (`wel6_q`). Every one of those columns came from
the one backward solve.

## Read the capture fractions

Read the `wel6_q` sensitivity at the prediction well cell for each measure. Sum
the per-period sensitivities over the periods the well actually pumps, 12 through
21, because each period's value is that period's contribution to the measure at
the final period. Print the baseline flow through each outlet alongside, so the
fractions can be compared with what each feature carries.

In [ ]:
active = range(11, nper)  # zero-based periods the prediction well pumps
capture = {}
for measure in measures:
    capture[measure] = adjh.total_sensitivity(
        ws, measure, "wel6_q", cell=pred_cell, periods=active
    )

labels = {"swgw": "stream", "lakegw": "lake", "coastgw": "coast"}
terms = {"swgw": "SFR(SFR-1)", "lakegw": "LAK(LAK-1)", "coastgw": "GHB(GHB-1)"}
print(f"outlet   capture fraction   baseline net flow, {unit('flow', plain=True)}")
for measure, label in labels.items():
    net = to_display(adjh.budget_net(ws, terms[measure])[last], "flow")
    print(f"{label:8s} {capture[measure]:12.4f}       {net:16,.0f}")
print(f"{'total':8s} {sum(capture.values()):12.4f}")

**What to look for.** The well takes 0.618 of its water from the sea and 0.402
from the stream; both are negative because the well rate is itself negative, and
a higher pumping rate makes each feature give up more water. The lake's 0.010
has the other sign: with the well pumping, the lake gives up less water to the
aquifer than before. The three together are -1.009, slightly more than all of
the pumping; the two-run check at the end shows where the extra goes.

The baseline flows show that neither the size nor the direction of a flow says
much about what a new well will draw on. The aquifer discharges 32,236 m³/d
(1,138,391 ft³/d) to the stream, the lake leaks 3,692 m³/d (130,376 ft³/d) to
the aquifer, and the sea is a net source of 4,071 m³/d (143,763 ft³/d) to the
aquifer rather than an outlet at all, and the sea still supplies most of this
well's water. Distance decides it: the coast is 762 m (2,500 ft) away, the
nearest stream reach 1,067 m (3,500 ft), and the lake 3,002 m (9,849 ft), which
is far enough to be out of reach.

### Map capture for a well anywhere in the aquifer

The `wel6_q` sensitivity is not only defined at the prediction well. mf6adj
returns it in every cell, so the same three backward solves say what fraction of
a well's pumping each outlet would supply wherever the well were placed. Map the
negative of the composite `wel6_q` sensitivity for each measure, which is the
capture fraction for a well pumping over every stress period, in layer 5, the
layer the prediction well is in, and compare it with layer 1.

A negative capture is a feature giving up less water to the aquifer under
pumping than it did before. The stream and coast maps share a color scale that
starts at zero, and the lake map is scaled to its own range, which is about five
times smaller and all negative.

In [ ]:
layer = pred_cell[0]  # zero-based: the layer the prediction well is in
capture_maps = {
    measure: -adjh.composite_sensitivity(ws, measure, "wel6_q") for measure in measures
}
panels = {"swgw": "a", "lakegw": "b", "coastgw": "c"}
active_cells = gwf.dis.idomain.array[layer] > 0
top_cells = gwf.dis.idomain.array[0] > 0
# the stream and coast share a scale from zero; the lake is scaled to itself
shared = np.concatenate(
    [capture_maps[m][layer][active_cells] for m in ("swgw", "coastgw")]
)
lake = capture_maps["lakegw"][layer][active_cells]
limits = {
    "swgw": (min(0.0, shared.min()), shared.max()),
    "coastgw": (min(0.0, shared.min()), shared.max()),
    "lakegw": (lake.min(), max(0.0, lake.max())),
}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [list(panels.values())], figsize=(11, 6), layout="constrained"
    )
    for measure, panel in panels.items():
        capture_map = capture_maps[measure]
        vmin, vmax = limits[measure]
        ax = axd[panel]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=layer)
        cb = mm.plot_array(capture_map[layer], cmap="viridis", vmin=vmin, vmax=vmax)
        mm.plot_ibound()

        # the stream, lake and coast are in layer 1, so outline their cells
        # rather than plotting them with plot_bc
        for cells, color in (
            (sfr_cells, "tab:cyan"),
            (lak_cells, "white"),
            (coast_cells, "tab:green"),
        ):
            rc = sorted({(i, j) for _, i, j in cells})
            ax.plot(
                [xc[i, j] for i, j in rc],
                [yc[i, j] for i, j in rc],
                "s",
                mfc="none",
                mec=color,
                ms=4,
            )
        ax.plot(px, py, "r*", ms=14)
        ax.xaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
        ax.yaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
        ax.set_xlabel(f"x, in {length_name()}")
        flopy.plot.styles.heading(ax=ax, letter=panel, heading=labels[measure])
        fig.colorbar(cb, ax=ax, shrink=0.4, label="capture fraction, dimensionless")
    axd["a"].set_ylabel(f"y, in {length_name()}")

for measure, label in labels.items():
    layer_map = capture_maps[measure][layer]
    difference = np.abs(capture_maps[measure][0] - layer_map)[top_cells]
    print(
        f"{label:8s} layer {layer + 1}: "
        f"{layer_map[active_cells].min():7.4f} to {layer_map[active_cells].max():7.4f}"
        f"   layers 1 and {layer + 1} differ by "
        f"{np.percentile(difference, 95):.4f} at 95% of cells, "
        f"at most {difference.max():.4f}"
    )

**What to look for.** Panel c is brightest along the southern row, where a well
takes up to 0.979 of its water from the sea, and fades north to 0.124. Panel a is
brightest over the north of the valley and along the stream, up to 0.944, and
falls toward the coast, to 0.022 on the southern row. The two are complementary:
every well is somewhere on a gradient between them.

Panel b is negative everywhere, from -0.0003 near the coast to -0.197 beneath the
lake. The lake has no outlet, so the water a well draws from it in one year is
not replaced: the stage falls, the lake shrinks, and it leaks less to the aquifer
in every later year. Added over all the years of pumping, the lowered stage
outweighs the extra leakage, so a well near the lake takes less water from it,
not more.

The depth of the well matters little. Layers 1 and 5 differ by 0.046 or less at
95 percent of the cells for the stream, 0.101 for the lake, and 0.015 for the
coast, and by at most 0.165, so how deep the well is set changes its capture far
less than where it is drilled.

### Where the sea replaces the stream as the main source

Read the two capture maps down the column the prediction well sits in, from the
northern end of the valley to the coast, to find where the sea begins to supply
more than the stream. Plot both fractions and their sum with distance from the
coast, and count how much of the valley each outlet dominates.

In [ ]:
col = pred_cell[2]
rows = [i for i in range(nrow) if active_cells[i, col]]
distance = to_display(np.array([yc[i, col] - yc[nrow - 1, col] for i in rows]))
profile = {
    measure: np.array([capture_maps[measure][layer, i, col] for i in rows])
    for measure in measures
}
total = sum(profile.values())

# the row where the two curves cross, read from the change of sign
gap = profile["coastgw"] - profile["swgw"]
crossing = np.flatnonzero(np.diff(np.sign(gap)) != 0)[0]
near, far = sorted(distance[crossing : crossing + 2])
print(
    f"the sea overtakes the stream between rows {rows[crossing] + 1} and "
    f"{rows[crossing + 1] + 1}, {near:,.0f} to {far:,.0f} {unit(plain=True)} inland"
)
print(
    f"all three outlets together supply {total.min():.3f} to {total.max():.3f} "
    f"down this column, and {total[rows.index(pred_cell[1])]:.3f} at the well"
)

winner = np.argmax(np.stack([capture_maps[m][layer] for m in measures]), axis=0)
for index, measure in enumerate(measures):
    share = (winner == index)[active_cells].mean()
    print(f"{labels[measure]:8s} supplies the most water at {share:5.1%} of the valley")

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
    for measure, color in (
        ("coastgw", "tab:green"),
        ("swgw", "tab:cyan"),
        ("lakegw", "tab:blue"),
    ):
        ax.plot(distance, profile[measure], color=color, lw=2, label=labels[measure])
    ax.plot(distance, total, "k--", lw=1.2, label="all three")
    well_distance = to_display(py - yc[nrow - 1, col])
    ax.axvline(well_distance, color="0.4", lw=0.8)
    ax.axhline(0.0, color="k", lw=0.6)
    ymin = min(0.0, min(p.min() for p in profile.values()))
    ymax = max(1.0, total.max())
    ax.text(
        well_distance,
        ymin + 0.02,
        " prediction well",
        fontsize=8,
        rotation=90,
        va="bottom",
    )
    ax.set_xlabel(f"distance inland from the coast, in {length_name()}")
    ax.set_ylabel("capture fraction, dimensionless")
    ax.set_xlim(0, distance.max())
    ax.set_ylim(ymin, ymax + 0.1)
    ax.legend(loc="upper center", ncols=4, fontsize=8)
    ax.set_title(f"Capture down column {col + 1}, layer {layer + 1}")

**What to look for.** The sea begins to supply more than the stream between rows
32 and 33, 1,067 to 1,219 m (3,500 to 4,000 ft) inland, and the prediction well
sits 762 m (2,500 ft) inland, on the seaward side of that divide. Over the whole
of layer 5 the stream supplies the most water at 82.3 percent of the valley and
the sea at 17.7 percent, and the lake supplies the most nowhere.

The dashed curve is what all three outlets supply together: 0.971 to 1.016 down
this column, and 1.011 at the well. Near the well the outlets supply a little
more than the pumping, and the two-run check at the end shows the extra goes to
make up recharge lost to evapotranspiration above the lowered water table.

## What 0.3 m of sea-level rise does

The same backward solves returned the sensitivity of each measure to the coastal
boundary itself: `ghb-1_bhead` is the derivative with respect to the boundary
head, and `ghb-1_cond` the derivative with respect to its conductance. A rise in
sea level is a rise in that boundary head, so `ghb-1_bhead` answers a question no
capture fraction can: how much more seawater the coast delivers, and how much
more water the stream receives, when sea level rises.

The equivalent freshwater head rises by the density ratio, 1.0245, times the rise
in sea level. Multiply the sensitivity by that rise for 0.3 m (0.98 ft) of
sea-level rise, sum it by layer to see which part of the coast carries the
response, and weight the conductance sensitivity by the conductance itself to
find how much a 1 percent error in the conductance would change the flows.

In [ ]:
sea_level_rise = 0.3  # m
# the equivalent freshwater head rises by the density ratio times the sea
density_ratio = (
    FRESHWATER_DENSITY + DENSITY_SLOPE * SEAWATER_CONCENTRATION
) / FRESHWATER_DENSITY
bhead_rise = density_ratio * sea_level_rise

cond = np.zeros((nlay, nrow, ncol))
for record in ghb_spd:
    cond[tuple(record["cellid"])] = record["cond"]

# change in each outlet's flow for the sea-level rise, cell by cell
rise_response = {
    measure: bhead_rise * adjh.composite_sensitivity(ws, measure, "ghb-1_bhead")
    for measure in measures
}
fu = unit("flow", plain=True)
print(
    f"sea-level rise {to_display(sea_level_rise):.2f} {unit(plain=True)}, "
    f"boundary-head rise {to_display(bhead_rise):.3f} {unit(plain=True)}"
)
print(f"outlet   change for the rise, {fu}   change for 1% more conductance, {fu}")
for measure, label in labels.items():
    dcond = adjh.composite_sensitivity(ws, measure, "ghb-1_cond")
    print(
        f"{label:8s} {to_display(np.nansum(rise_response[measure]), 'flow'):16,.0f}"
        f" {to_display(0.01 * np.nansum(cond * dcond), 'flow'):30,.0f}"
    )
deep = np.nansum(rise_response["coastgw"][3:]) / np.nansum(rise_response["coastgw"])
print(
    f"\nlayers 4 and 5 hold {cond[3:].sum() / cond.sum():.1%} of the coastal "
    f"conductance and carry {deep:.1%} of the coastal response"
)

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
    width = 0.38
    offsets = {"coastgw": -0.5 * width, "swgw": 0.5 * width}
    for measure, color in (("coastgw", "tab:green"), ("swgw", "tab:cyan")):
        by_layer = [
            to_display(np.nansum(rise_response[measure][k]), "flow")
            for k in range(nlay)
        ]
        ax.bar(
            np.arange(1, nlay + 1) + offsets[measure],
            by_layer,
            width,
            color=color,
            edgecolor="k",
            lw=0.4,
            label=labels[measure],
        )
    ax.axhline(0.0, color="k", lw=0.8)
    ax.set_xlabel("model layer")
    ax.set_ylabel(f"change in flow, in {unit('flow')}")
    ax.set_xticks(range(1, nlay + 1))
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title(
        f"Response of each outlet to {to_display(sea_level_rise):.2f} "
        f"{unit()} of sea-level rise"
    )

**What to look for.** 0.3 m (0.98 ft) of sea-level rise raises the boundary head
0.307 m (1.008 ft). It brings in 4,665 m³/d (164,760 ft³/d) more water across the
coast and sends 4,840 m³/d (170,915 ft³/d) more into the stream, which is 15
percent of the stream's 32,236 m³/d (1,138,391 ft³/d) baseline. The extra water
that enters leaves through the stream: a higher sea raises the heads inland, and
the stream is where the aquifer discharges them. The lake's leakage changes by
54 m³/d (1,908 ft³/d).

Layers 4 and 5 hold 88.9 percent of the coastal conductance and carry 89.9
percent of the response, so a coastal boundary is only as good as the deep
conductance behind it. A 1 percent error in that conductance changes the coastal
flow and the stream by 12 m³/d (407 ft³/d), which is 0.04 percent of the
stream's baseline, so the boundary head is the term worth getting right.

## Which parts of the aquifer control each outlet

The same backward solves also returned the sensitivity of each measure to
hydraulic conductivity in every cell. Map the composite `k11` sensitivity for the
stream and the coast in layer 5, and correlate the two patterns. The sensitivity
is a change in flow per unit of conductivity.

In [ ]:
# the change in flow per unit of conductivity is a flow per rate, an area
k11_sens = {
    measure: to_display(adjh.composite_sensitivity(ws, measure, "k11"), "area")
    for measure in ("swgw", "coastgw")
}
pair = [k11_sens[m][layer][active_cells] for m in ("swgw", "coastgw")]
print(f"correlation between the two patterns: {np.corrcoef(*pair)[0, 1]: .3f}")
for measure, values in zip(("swgw", "coastgw"), pair):
    ranked = np.sort(np.abs(values))[::-1]
    share = np.cumsum(ranked) / ranked.sum()
    print(
        f"{labels[measure]:8s} largest 10% of cells carry "
        f"{share[int(0.1 * ranked.size)]:.2f} of the total"
    )
au = unit("area", plain=True)
for measure in ("swgw", "coastgw"):
    strongest = np.unravel_index(
        np.argmax(np.abs(k11_sens[measure][layer])), (nrow, ncol)
    )
    print(
        f"strongest cell for the {labels[measure]}: row {strongest[0] + 1}, "
        f"column {strongest[1] + 1}, worth "
        f"{k11_sens['swgw'][layer][strongest]:9,.1f} {au} to the stream and "
        f"{k11_sens['coastgw'][layer][strongest]:9,.1f} {au} to the coast"
    )

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic([["a", "b"]], figsize=(8, 6.5), layout="constrained")
    for measure, panel in (("swgw", "a"), ("coastgw", "b")):
        arr = k11_sens[measure][layer]
        vmax = np.percentile(np.abs(arr[active_cells]), 99)
        ax = axd[panel]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=layer)
        cb = mm.plot_array(arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        mm.plot_ibound()
        ax.plot(px, py, "k*", ms=14)
        ax.xaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
        ax.yaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
        ax.set_xlabel(f"x, in {length_name()}")
        flopy.plot.styles.heading(ax=ax, letter=panel, heading=labels[measure])
        fig.colorbar(
            cb,
            ax=ax,
            shrink=0.4,
            label=f"change in flow per {unit('rate')}, in {unit('flow')}",
        )
    axd["a"].set_ylabel(f"y, in {length_name()}")

**What to look for.** The two maps are close to mirror images, with a
correlation of -0.958. The strongest cell for both is in row 40, column 9,
beneath the last stream reach, where the stream leaves the valley. Its
sensitivity is -3.3 m³/d per m/d (-35.3 ft³/d per ft/d) for the stream measure
and 3.2 m³/d per m/d (34.5 ft³/d per ft/d) for the coast.

The signs differ because the baselines do, not because the two outlets compete.
The stream takes water out of the aquifer and the sea puts it in, so conductivity
in the cells between them moves more water along the same path from the sea to
the stream, and both flows grow at once. The cells that control the coast are the
cells that control the stream.

For both measures, the tenth of the cells with the largest sensitivity carry 0.59
of the total, so the adjoint shows where the next field measurement would be
most useful, and it names nearly the same cells for both.

## Check the adjoint with two model runs

The direct way to get a capture fraction is to run the model twice, once with
the well off and once with it on, and difference the budgets. That gives one
number per run, and it is an independent check on the adjoint.

Run the model again with the prediction well pumping at a small rate. Keep the
rate small: the adjoint returns a derivative evaluated at zero pumping, so the
difference only matches it in the limit of a small perturbation. Tighten the
solver's head-change criterion at the same time, because the differences in lake
flow are small enough to be lost within the solver's convergence tolerance.
Difference every budget term rather than only the three measures, so the whole
of the pumping is accounted for, and list the periods the seepage drains flow
in. Save the unsaturated-zone budget as well (`uzf_budget=True`), to see what
pumping changes above the water table, and read the final lake stage from the
lake observations.

In [ ]:
dq = -100.0  # m^3/d, small enough to stay in the linear range

ws_base = adjh.prepare_model(
    "adj-coastal-q0",
    variant="coastal-advanced",
    prediction_rate=0.0,
    equivalent_freshwater=True,
    outer_dvclose=1.0e-10,
    uzf_budget=True,
)
ws_pert = adjh.prepare_model(
    "adj-coastal-dq",
    variant="coastal-advanced",
    prediction_rate=dq,
    equivalent_freshwater=True,
    outer_dvclose=1.0e-10,
    uzf_budget=True,
)
for w in (ws_base, ws_pert):
    adjh.run_model(w, mf6_exe)

print("outlet     adjoint   two-run   difference")
for measure, label in labels.items():
    diff = (
        adjh.budget_net(ws_pert, terms[measure])[last]
        - adjh.budget_net(ws_base, terms[measure])[last]
    ) / dq
    print(
        f"{label:8s} {capture[measure]:9.4f} {diff:9.4f} "
        f"{100 * abs(diff - capture[measure]) / abs(capture[measure]):9.1f}%"
    )

print("\nshare of the pumping each budget term supplies in the final period")
sources = adjh.pumping_sources(ws_base, ws_pert, dq, last)
for term, share in sources.items():
    print(f"  {term:20s} {share:8.4f}")
print(f"  {'total':20s} {sum(sources.values()):8.4f}")

# seepage reaches the drains only when the water table is near the land surface
drained = -(
    adjh.budget_net(ws_base, "DRN(DRN-1)")
    + adjh.budget_net(ws_base, "DRN-TO-MVR(DRN-1)")
)
wet = {kper + 1: q for kper, q in enumerate(drained) if q > 0.01}
print("\nseepage-drain outflow without the well, by stress period")
for kper, q in wet.items():
    print(f"  {kper:2d} {to_display(q, 'flow'):10,.0f} {unit('flow', plain=True)}")
if not wet:
    print("  none in any period")

# what pumping changes in the unsaturated zone and at the lake
print("\nchange in the unsaturated-zone budget per unit pumped, final period")
uzf_base = adjh.csv_row(ws_base, "sv.uzf.csv", last)
uzf_pert = adjh.csv_row(ws_pert, "sv.uzf.csv", last)
for column in uzf_base.index:
    if column in ("time", "PERCENT_DIFFERENCE") or column.startswith("TOTAL"):
        continue
    if uzf_base[column] != 0.0 or uzf_pert[column] != 0.0:
        print(f"  {column:18s} {(uzf_pert[column] - uzf_base[column]) / -dq:8.4f}")
stage = [
    to_display(adjh.csv_row(w, "sv.lake.obs.csv", last)["LAKE-STAGE"])
    for w in (ws_base, ws_pert)
]
print(
    f"\nfinal lake stage: {stage[0]:.3f} {unit(plain=True)} without the well, "
    f"{stage[1]:.3f} {unit(plain=True)} with it"
)

**What to look for.** The stream agrees with the two-run difference to 0.5
percent, the coast to 2.8 percent, and the lake to 0.7 percent. Repeating the
pair of runs at a tenth and at ten times this rate changes the differences by
less than 0.001, so the gaps are not linearization errors.

The budget decomposition says what the gap is. Every unit the well pumps in the
final year is supplied by the sea (0.6349), the stream (0.4040), and aquifer
storage (0.0057: 0.0032 from the aquifer's elastic storage and 0.0025 from
draining at the water table), less the 0.0103 the lake no longer gives up and the 0.0343 less
recharge UZF delivers to the water table, and those add to 1.0000. The
unsaturated-zone budget shows where that recharge goes: infiltration does not
change, evapotranspiration from the unsaturated zone rises by 0.0348 of the
pumping, and recharge to the water table falls by 0.0343. This model takes
evapotranspiration only from the unsaturated zone, within 3.048 m (10 ft) of the
land surface, so a lower water table leaves more unsaturated ground within reach
of the roots. The sea and the stream make up that loss as well as the pumping.
mf6adj reported at the start of the adjoint solve that it forms no terms for UZF,
so the adjoint leaves out the recharge term, which accounts for part of the
coast's gap, 0.017.

The lake's final stage is 4.536 m (14.882 ft) without the well and 4.534 m
(14.877 ft) with it: the small drop that makes the lake leak less. The lake
terms are under 3 percent of the stream's either way, so this well captures water
from the sea and the stream. The seepage drains flow only in period 1, the
steady spin-up, at 50 m³/d (1,764 ft³/d), so they supply none of the well's water.

An adjoint sensitivity answers the question its formulation encodes, and mf6adj
reports which parts of that formulation are approximate. Where the difference
changes the answer, the two-run difference is the number to trust.

## Recap

- A **performance measure** names the model output you care about. Three
  measures, one per outlet, came out of a single forward run and three backward
  solves.
- A saltwater coast in a constant-density model is a boundary at the
  **equivalent freshwater head**, which rises from 0.019 m (0.061 ft) in layer 1
  to 1.76 m (5.77 ft) at the bottom of the valley.
- The prediction well captures 0.618 of its pumping from the sea and 0.402 from
  the stream, and the lake gives up 0.010 of it less than before. Distance to the
  outlet decides the split, not the size of the outlet.
- Mapped over the grid, the same `wel6_q` sensitivity gives the capture fraction
  for a well placed anywhere. The sea supplies more than the stream on the
  seaward side of a divide 1,067 to 1,219 m (3,500 to 4,000 ft) inland on the
  well's column, and over 17.7 percent of the valley.
- The sensitivity to the boundary head is the sea-level-rise question: 0.3 m
  (0.98 ft) brings in 4,665 m³/d (164,760 ft³/d) more seawater and sends 4,840
  m³/d (170,915 ft³/d) more to the stream, 89.9 percent of it through the two
  deep layers.
- Conductivity that makes the stream gain more makes the sea deliver more, with a
  correlation of -0.958 between the two sensitivity patterns.
- mf6adj warns when part of a package's derivative is not formed. Read those
  warnings, and check a measure with a small-perturbation two-run difference
  and a full water budget when they name something the model uses. Here the
  evapotranspiration UZF takes above a lowered water table is what the coast
  measure leaves out, an error of under 3 percent.